# Prepare AffectNet (Kaggle copy) for SMIRK

Requires an AffectNet copy with class folders `0..7` (optionally inside `train/ val/ test/`) added as input (*Add Input*).
Output: `/kaggle/working/affectnet_crops/` (`images/*.png` + `meta.csv`), same layout as the ExpW crops.

## 1. Locate the dataset

In [ ]:
import glob, os

# folders that directly contain the class folders 0..7: either <root>/0..7 or <root>/{train,val,test}/0..7
class_dirs = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/7', recursive=True)
                     if os.path.isdir(os.path.join(os.path.dirname(p), '0'))})
print('class folders found in:', class_dirs)
assert class_dirs, 'no folder with class folders 0..7 found - check the input dataset'

split_parents = {os.path.dirname(d) for d in class_dirs if os.path.basename(d) in ('train', 'val', 'test')}
INPUT_DIR = split_parents.pop() if len(split_parents) == 1 else class_dirs[0]
assert len(class_dirs) == 1 or os.path.dirname(class_dirs[0]) == INPUT_DIR, 'several AffectNet copies found - keep only one input'
print('INPUT_DIR =', INPUT_DIR)

for d in class_dirs:
    print(os.path.relpath(d, INPUT_DIR), {c: len(os.listdir(os.path.join(d, c))) for c in sorted(os.listdir(d))})

## 2. Check the label mapping visually

Expected (FER2013 order + contempt): **0 angry, 1 disgust, 2 fear, 3 happy, 4 sad, 5 surprise, 6 neutral, 7 contempt**.
Each row below shows 8 random images of one class folder. If a row does not match its name, tell me before continuing.

In [ ]:
import random, cv2, matplotlib.pyplot as plt

EXPRESSIONS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral', 'contempt']
SAMPLE_DIR = os.path.join(INPUT_DIR, 'test') if os.path.isdir(os.path.join(INPUT_DIR, 'test')) else INPUT_DIR
random.seed(0)
fig, axes = plt.subplots(8, 8, figsize=(16, 17))
for c in range(8):
    folder = os.path.join(SAMPLE_DIR, str(c))
    for ax, f in zip(axes[c], random.sample(os.listdir(folder), 8)):
        ax.imshow(cv2.imread(os.path.join(folder, f))[..., ::-1]); ax.axis('off')
    axes[c][0].set_title(f'{c}: {EXPRESSIONS[c]}', loc='left', fontsize=14)
plt.tight_layout(); plt.show()

## 3. Write the conversion script

In [ ]:
%%writefile /kaggle/working/prepare_affectnet.py
"""
Convert a folder-structured AffectNet copy into the same layout as the ExpW crops (crop_expw.py).

Expected input layout (AffectNet copies found on Kaggle, where faces are already cropped), either
    input_dir/{train,val,test}/{0..7}/*.jpg     (split is kept)
or
    input_dir/{0..7}/*.jpg                      (split is set to 'unsplit')

The folder numbers follow the FER2013 order (same as ExpW) plus contempt:
    0 angry, 1 disgust, 2 fear, 3 happy, 4 sad, 5 surprise, 6 neutral, 7 contempt
This mapping was inferred by looking at samples and at the class sizes (disgust 3803 and contempt 3750 train images,
as in the official AffectNet), so check it visually before using the labels (see kaggle_prepare_affectnet.ipynb).

Each image is resized to --image_size x --image_size (the images are square face crops; the few non-square ones are
padded with black first) and saved as PNG.

Output:
    out_dir/images/{split}_{label}_{idx:05d}.png
    out_dir/meta.csv     name, split, label, expression, source file, original width/height
"""
import os
import csv
import argparse
import numpy as np
import cv2
from tqdm import tqdm

EXPRESSIONS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral', 'contempt']


def to_square(image):
    h, w = image.shape[:2]
    if h == w:
        return image
    size = max(h, w)
    top, left = (size - h) // 2, (size - w) // 2
    return cv2.copyMakeBorder(image, top, size - h - top, left, size - w - left, cv2.BORDER_CONSTANT, value=0)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Convert AffectNet (folder per class) to the ExpW-crops layout.')
    parser.add_argument('--input_dir', type=str, required=True, help='Directory containing train/ val/ test/')
    parser.add_argument('--out_dir', type=str, required=True, help='Output directory')
    parser.add_argument('--image_size', type=int, default=256, help='Size of the square output images')
    parser.add_argument('--min_face_size', type=float, default=160, help='Skip images smaller than this (in pixels)')
    parser.add_argument('--skip_contempt', action='store_true', help='Drop class 7 (contempt is not in ExpW)')
    args = parser.parse_args()

    os.makedirs(os.path.join(args.out_dir, 'images'), exist_ok=True)

    # some copies have split folders (input_dir/{train,val,test}/{0..7}), others only class folders (input_dir/{0..7});
    # the latter are marked 'unsplit' and split later
    splits = [s for s in ['train', 'val', 'test'] if os.path.isdir(os.path.join(args.input_dir, s))]
    sources = [(s, os.path.join(args.input_dir, s)) for s in splits] or [('unsplit', args.input_dir)]

    files = []
    for split, split_dir in sources:
        for label in range(len(EXPRESSIONS)):
            folder = os.path.join(split_dir, str(label))
            if not os.path.isdir(folder) or (args.skip_contempt and label == 7):
                continue
            for filename in sorted(os.listdir(folder)):
                if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    files.append((split, label, os.path.join(folder, filename)))

    rows = []
    skipped = {'too_small': 0, 'unreadable': 0}
    for idx, (split, label, path) in enumerate(tqdm(files)):
        image = cv2.imread(path)
        if image is None:
            skipped['unreadable'] += 1
            continue

        h, w = image.shape[:2]
        if min(h, w) < args.min_face_size:
            skipped['too_small'] += 1
            continue

        image = cv2.resize(to_square(image), (args.image_size, args.image_size), interpolation=cv2.INTER_AREA)

        name = f'{split}_{label}_{idx:05d}'
        cv2.imwrite(os.path.join(args.out_dir, 'images', name + '.png'), image)
        rows.append([name, split, label, EXPRESSIONS[label], os.path.relpath(path, args.input_dir), w, h])

    with open(os.path.join(args.out_dir, 'meta.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['name', 'split', 'label', 'expression', 'source_file', 'orig_width', 'orig_height'])
        writer.writerows(rows)

    print(f'Saved {len(rows)} images, skipped {skipped}')
    for split, _ in sources:
        counts = np.bincount([r[2] for r in rows if r[1] == split], minlength=len(EXPRESSIONS))
        print(f'{split}:', dict(zip(EXPRESSIONS, counts.tolist())))

## 4. Convert

Images are resized to 256x256 PNG; images smaller than 160 px are skipped.

In [ ]:
!python /kaggle/working/prepare_affectnet.py --input_dir "$INPUT_DIR" --out_dir /kaggle/working/affectnet_crops --min_face_size 160

## 5. Look at some converted images

In [ ]:
import pandas as pd

meta = pd.read_csv('/kaggle/working/affectnet_crops/meta.csv')
print(pd.crosstab(meta['expression'], meta['split']))

sample = meta.sample(24, random_state=0)
fig, axes = plt.subplots(4, 6, figsize=(15, 10))
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    ax.imshow(cv2.imread(f"/kaggle/working/affectnet_crops/images/{row['name']}.png")[..., ::-1])
    ax.set_title(f"{row['split']} / {row['expression']}"); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Save the result

*Save Version → Save & Run All*. When it finishes, open the **Output** tab and choose **New Dataset**
(for example `affectnet-crops`, Private).